# ADJSCC on CIFAR-10 — Colab training

Reproduces the core claim of
[arXiv:2012.00533](https://arxiv.org/abs/2012.00533): **one** SNR-adaptive
model matches or beats a whole ensemble of fixed-SNR models across the SNR
range, with no retraining.

Same six-arm shape as the speech notebook, so the two are directly
comparable. Fidelity notes live in `docs/GAPS.md` and
`docs/CODE_VS_PAPER.md` — read the latter before quoting any number.

## 1. Setup

Clones the repo and installs dependencies when running on Colab; a no-op
if you already opened this notebook inside a local checkout.

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = os.environ.get('ADJSCC_REPO', 'https://github.com/prashantrajbista/wireless_image_transmission.git')

if IN_COLAB:
    if not pathlib.Path('wireless_image_transmission').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
    os.chdir('wireless_image_transmission')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'soundfile', 'pystoi', 'pesq', 'pytorch-msssim',
                    'huggingface_hub', 'pyarrow', 'wandb'], check=True)
else:
    # running from notebooks/ inside a checkout
    if pathlib.Path.cwd().name == 'notebooks':
        os.chdir('..')

sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
import torch
from adjscc.engine import default_args, train, pick_device

DEVICE = pick_device()
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cpu':
    print('WARNING: no GPU. On Colab: Runtime > Change runtime type > GPU.')

# DataLoader workers: Colab (Linux, fork) is happy with 2; notebooks on
# macOS/Windows use spawn and need 0 or they deadlock on import.
WORKERS = 2 if sys.platform.startswith('linux') else 0

### Weights & Biases (optional)

Skip this cell to train without logging — `train()` degrades to printing.
Every run records its full dataset provenance (corpus, source, sample rate,
clip length, how many clips were dropped) alongside the usual hyperparameters,
so two runs can be checked for comparability from the config alone.

In [ ]:
USE_WANDB = False  # set True and run this cell to enable logging

if USE_WANDB:
    import wandb
    wandb.login()  # paste your key when prompted
    os.environ.setdefault('WANDB_PROJECT', 'wireless-image-transmission')

## 2. Configuration

CIFAR-10 downloads automatically on first use. Defaults are the paper's;
`EPOCHS` is the one to cut first — the trend shows up long before 1280.

In [ ]:
RATIO = 1/6      # R = C/96, so C=16 — the paper's headline setting
FILTERS = 256    # paper value; 64 trains much faster
EPOCHS = 100     # paper uses 1280
BATCH = 128

## 3. Sanity check

In [ ]:
sanity = default_args(
    modality='image', cond='af', ratio=RATIO,
    filters=64, epochs=2, batch=BATCH, workers=WORKERS,
    out='ckpt/img_sanity.pt', no_wandb=True)
train(sanity)

In [ ]:
import torch, numpy as np
import matplotlib.pyplot as plt
from adjscc.engine import load_model
from adjscc.data import loaders

model, ck = load_model('ckpt/img_sanity.pt', DEVICE)
_, test_loader = loaders(8, workers=0)
x, _ = next(iter(test_loader))
x = x[:6].to(DEVICE)
with torch.no_grad():
    recs = {snr: model(x, snr).cpu().numpy() for snr in (0.0, 10.0, 20.0)}

rows = [('original', x.cpu().numpy())] + [(f'{s:g} dB', r) for s, r in recs.items()]
fig, axes = plt.subplots(len(rows), 6, figsize=(9, 1.6 * len(rows)))
for ax_row, (label, imgs) in zip(axes, rows):
    ax_row[0].set_ylabel(label, fontsize=9)
    for ax, im in zip(ax_row, imgs):
        ax.imshow(np.clip(im.transpose(1, 2, 0), 0, 1))
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 4. The six arms

`af` and `reg` are the two SNR-conditioning mechanisms; the fixed-SNR
BDJSCC runs are the paper's Fig. 6 baselines. Identical budget across all
arms or the comparison means nothing.

In [ ]:
RUNS = [
    dict(name='img_af',    cond='af',   snr_fixed=None),
    dict(name='img_reg',   cond='reg',  snr_fixed=None),
    dict(name='img_bd1',   cond='none', snr_fixed=1.0),
    dict(name='img_bd7',   cond='none', snr_fixed=7.0),
    dict(name='img_bd13',  cond='none', snr_fixed=13.0),
    dict(name='img_bd19',  cond='none', snr_fixed=19.0),
]

results = {}
for r in RUNS:
    out = f"ckpt/{r['name']}.pt"
    if pathlib.Path(out).exists():
        print(f"skip {r['name']} (checkpoint exists)")
        continue
    print(f"\n=== {r['name']} " + '=' * 40)
    args = default_args(
        modality='image', cond=r['cond'], snr_fixed=r['snr_fixed'],
        ratio=RATIO, filters=FILTERS, epochs=EPOCHS, batch=BATCH,
        workers=WORKERS, out=out, no_wandb=not USE_WANDB)
    results[r['name']] = train(args)

results

## 5. Evaluate

In [ ]:
REPEATS = 1        # paper uses 10
SNRS = list(range(0, 21, 2))

for r in RUNS:
    ckpt = f"ckpt/{r['name']}.pt"
    if not pathlib.Path(ckpt).exists():
        continue
    subprocess.run([sys.executable, 'scripts/eval.py', '--ckpt', ckpt,
                    '--snr-list', ','.join(map(str, SNRS)),
                    '--repeats', str(REPEATS),
                    '--out', f"results/{r['name']}.csv"], check=True)

## 6. The paper's Fig. 6

Each fixed-SNR baseline peaks near its training SNR and falls away on both
sides. The adaptive curves should envelope them — one model covering what
the ensemble needed four for.

In [ ]:
import csv

def load(name, col='psnr'):
    p = pathlib.Path(f'results/{name}.csv')
    if not p.exists():
        return None
    with open(p) as f:
        rows = list(csv.DictReader(f))
    return [float(r['snr']) for r in rows], [float(r[col]) for r in rows]

STYLE = {'img_af': ('ADJSCC (pooled feats + SNR)', 'o-', 2.2),
         'img_reg': ('ReJSCC (SNR only)', 's-', 2.2),
         'img_bd1': ('BDJSCC @1 dB', ':', 1.0),
         'img_bd7': ('BDJSCC @7 dB', ':', 1.0),
         'img_bd13': ('BDJSCC @13 dB', ':', 1.0),
         'img_bd19': ('BDJSCC @19 dB', ':', 1.0)}

plt.figure(figsize=(7, 5))
for name, (label, fmt, lw) in STYLE.items():
    d = load(name)
    if d:
        plt.plot(d[0], d[1], fmt, label=label, linewidth=lw)
plt.xlabel('test SNR (dB)'); plt.ylabel('PSNR (dB)')
plt.title(f'CIFAR-10, R={RATIO:.4f}, AWGN')
plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig('graphs/image_psnr_vs_snr.png', dpi=150)
plt.show()